# Experiment 7 v2 — Universal Language Bootstrapper

Extends v1 (6 languages, 20 concepts, no quality gate) with:
- **10 languages** covering ~5B speakers including Mandarin, Hindi, Arabic, Russian
- **Cluster quality scoring** per proto-token (intra_variance, nn_margin, quality_score)
- **Universal Language Loss** with all three terms (L_proximity, L_coverage, L_complexity)
- **Compositionality probe**: semantic arithmetic in centroid space
- **Held-out concept set** for coverage measurement

See `research-design.md` for the full theoretical motivation.

In [ ]:
!pip install sentence-transformers umap-learn matplotlib scipy -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sentence_transformers import SentenceTransformer
from scipy.spatial.distance import cosine, pdist
from scipy.stats import spearmanr
import warnings
warnings.filterwarnings('ignore')

try:
    import umap
    HAS_UMAP = True
except ImportError:
    HAS_UMAP = False
    print('UMAP not available, skipping 2D projection')

model = SentenceTransformer('LaBSE')
print('LaBSE loaded')

In [ ]:
# ── Vocabulary: 10 languages, 20 TRAIN concepts + 20 HELD-OUT concepts ────────
#
# Speaker weights (millions of native speakers, approximate 2024 figures)
SPEAKER_WEIGHTS = {
    'en': 1500, 'zh': 1100, 'hi': 600,
    'es': 560,  'ar': 380,  'ru': 260,
    'pt': 260,  'fr': 280,  'de': 130, 'ja': 125,
}

# 20 training concepts — used to build proto-tokens
TRAIN_CONCEPTS = [
    'water', 'fire',  'earth', 'sky',   'love',
    'fear',  'trust', 'light', 'dark',  'time',
    'life',  'death', 'eat',   'sleep', 'run',
    'give',  'take',  'speak', 'think', 'feel',
]

# 20 held-out concepts — used ONLY for L_coverage (expressibility)
HELD_OUT_CONCEPTS = [
    'wind',   'stone',  'river',  'mountain', 'forest',
    'child',  'elder',  'friend', 'enemy',    'peace',
    'war',    'pain',   'joy',    'hope',     'dream',
    'build',  'break',  'find',   'lose',     'change',
]

MULTILINGUAL_VOCAB = {
    'en': {
        'train': ['water','fire','earth','sky','love','fear','trust','light','dark','time',
                  'life','death','eat','sleep','run','give','take','speak','think','feel'],
        'heldout': ['wind','stone','river','mountain','forest','child','elder','friend','enemy','peace',
                    'war','pain','joy','hope','dream','build','break','find','lose','change'],
    },
    'zh': {
        'train': ['水','火','土','天空','爱','恐惧','信任','光','暗','时间',
                  '生命','死亡','吃','睡觉','跑','给','拿','说话','思考','感觉'],
        'heldout': ['风','石头','河流','山','森林','孩子','老人','朋友','敌人','和平',
                    '战争','痛苦','喜悦','希望','梦想','建造','破坏','找到','失去','改变'],
    },
    'hi': {
        'train': ['पानी','आग','पृथ्वी','आकाश','प्यार','डर','विश्वास','रोशनी','अंधेरा','समय',
                  'जीवन','मृत्यु','खाना','सोना','दौड़ना','देना','लेना','बोलना','सोचना','महसूस करना'],
        'heldout': ['हवा','पत्थर','नदी','पहाड़','जंगल','बच्चा','बुजुर्ग','दोस्त','दुश्मन','शांति',
                    'युद्ध','दर्द','खुशी','आशा','सपना','बनाना','तोड़ना','ढूंढना','खोना','बदलना'],
    },
    'es': {
        'train': ['agua','fuego','tierra','cielo','amor','miedo','confianza','luz','oscuridad','tiempo',
                  'vida','muerte','comer','dormir','correr','dar','tomar','hablar','pensar','sentir'],
        'heldout': ['viento','piedra','río','montaña','bosque','niño','anciano','amigo','enemigo','paz',
                    'guerra','dolor','alegría','esperanza','sueño','construir','romper','encontrar','perder','cambiar'],
    },
    'ar': {
        'train': ['ماء','نار','أرض','سماء','حب','خوف','ثقة','ضوء','ظلام','وقت',
                  'حياة','موت','أكل','نوم','ركض','أعطى','أخذ','تكلم','فكر','شعر'],
        'heldout': ['ريح','حجر','نهر','جبل','غابة','طفل','مسن','صديق','عدو','سلام',
                    'حرب','ألم','فرح','أمل','حلم','بنى','كسر','وجد','فقد','تغير'],
    },
    'ru': {
        'train': ['вода','огонь','земля','небо','любовь','страх','доверие','свет','тьма','время',
                  'жизнь','смерть','есть','спать','бежать','давать','брать','говорить','думать','чувствовать'],
        'heldout': ['ветер','камень','река','гора','лес','ребёнок','старик','друг','враг','мир',
                    'война','боль','радость','надежда','мечта','строить','ломать','найти','потерять','изменить'],
    },
    'pt': {
        'train': ['água','fogo','terra','céu','amor','medo','confiança','luz','escuridão','tempo',
                  'vida','morte','comer','dormir','correr','dar','tomar','falar','pensar','sentir'],
        'heldout': ['vento','pedra','rio','montanha','floresta','criança','idoso','amigo','inimigo','paz',
                    'guerra','dor','alegria','esperança','sonho','construir','quebrar','encontrar','perder','mudar'],
    },
    'fr': {
        'train': ['eau','feu','terre','ciel','amour','peur','confiance','lumière','obscurité','temps',
                  'vie','mort','manger','dormir','courir','donner','prendre','parler','penser','ressentir'],
        'heldout': ['vent','pierre','rivière','montagne','forêt','enfant','aîné','ami','ennemi','paix',
                    'guerre','douleur','joie','espoir','rêve','construire','casser','trouver','perdre','changer'],
    },
    'de': {
        'train': ['Wasser','Feuer','Erde','Himmel','Liebe','Angst','Vertrauen','Licht','Dunkel','Zeit',
                  'Leben','Tod','essen','schlafen','laufen','geben','nehmen','sprechen','denken','fühlen'],
        'heldout': ['Wind','Stein','Fluss','Berg','Wald','Kind','Alter','Freund','Feind','Frieden',
                    'Krieg','Schmerz','Freude','Hoffnung','Traum','bauen','brechen','finden','verlieren','ändern'],
    },
    'ja': {
        'train': ['水','火','土','空','愛','恐怖','信頼','光','闇','時間',
                  '生命','死','食べる','寝る','走る','与える','取る','話す','考える','感じる'],
        'heldout': ['風','石','川','山','森','子供','老人','友達','敵','平和',
                    '戦争','痛み','喜び','希望','夢','建てる','壊す','見つける','失う','変える'],
    },
}

print(f'Languages: {list(MULTILINGUAL_VOCAB.keys())}')
print(f'Total speaker weight: {sum(SPEAKER_WEIGHTS.values()):,}M')
print(f'Train concepts: {len(TRAIN_CONCEPTS)}, Held-out: {len(HELD_OUT_CONCEPTS)}')

In [ ]:
# ── Embed all words ───────────────────────────────────────────────────────────
records = []
for lang, splits in MULTILINGUAL_VOCAB.items():
    for concept, word in zip(TRAIN_CONCEPTS, splits['train']):
        records.append({'lang': lang, 'concept': concept, 'word': word, 'split': 'train'})
    for concept, word in zip(HELD_OUT_CONCEPTS, splits['heldout']):
        records.append({'lang': lang, 'concept': concept, 'word': word, 'split': 'heldout'})

df = pd.DataFrame(records)
all_words = df['word'].tolist()
print(f'Embedding {len(all_words)} words across {len(MULTILINGUAL_VOCAB)} languages...')

all_embs = model.encode(all_words, normalize_embeddings=True,
                        batch_size=64, show_progress_bar=True)
df['emb'] = list(all_embs)
print('Done.')

In [ ]:
# ── Compute speaker-weighted centroids ───────────────────────────────────────
def compute_centroids(df, concept_list, split):
    """Speaker-weighted mean embedding per concept."""
    centroids = {}
    for concept in concept_list:
        sub = df[(df['concept'] == concept) & (df['split'] == split)]
        w   = np.array([SPEAKER_WEIGHTS[l] for l in sub['lang']], dtype=float)
        w  /= w.sum()
        vecs = np.stack(sub['emb'].values)      # (n_langs, 768)
        c    = (vecs * w[:, None]).sum(axis=0)
        c   /= np.linalg.norm(c)
        centroids[concept] = c
    return centroids

train_centroids  = compute_centroids(df, TRAIN_CONCEPTS,   'train')
heldout_centroids = compute_centroids(df, HELD_OUT_CONCEPTS, 'heldout')

centroid_matrix = np.stack([train_centroids[c] for c in TRAIN_CONCEPTS])
print(f'Centroid matrix shape: {centroid_matrix.shape}')

In [ ]:
# ── Cluster quality scoring ───────────────────────────────────────────────────
# For each proto-token:
#   intra_variance : mean cosine distance from each language's word to centroid
#   nn_margin      : cosine distance to nearest OTHER concept centroid
#   quality_score  : nn_margin / intra_variance  (higher = tighter + better separated)

def cosine_dist(a, b):
    return 1.0 - float(np.dot(a, b))   # both already unit-normalised

quality_rows = []
for i, concept in enumerate(TRAIN_CONCEPTS):
    centroid = train_centroids[concept]

    # intra_variance
    sub  = df[(df['concept'] == concept) & (df['split'] == 'train')]
    dists = [cosine_dist(centroid, v) for v in sub['emb'].values]
    intra_var = float(np.mean(dists))

    # nn_margin: distance to nearest OTHER centroid
    other_centroids = [train_centroids[c] for c in TRAIN_CONCEPTS if c != concept]
    nn_dist = float(min(cosine_dist(centroid, oc) for oc in other_centroids))

    quality = nn_dist / (intra_var + 1e-9)

    quality_rows.append({
        'concept'       : concept,
        'intra_variance': round(intra_var, 5),
        'nn_margin'     : round(nn_dist,   5),
        'quality_score' : round(quality,   3),
    })

quality_df = pd.DataFrame(quality_rows).sort_values('quality_score', ascending=False)
print(quality_df.to_string(index=False))

In [ ]:
# ── Universal Language Loss Function ─────────────────────────────────────────
# L_UL = λ1·L_proximity + λ2·L_coverage + λ3·L_complexity_reg
#
# L_proximity : speaker-weighted mean distance from centroid to each language's word
# L_coverage  : mean distance from held-out centroids to nearest proto-token
# L_complexity: 1 / H(pairwise_distances) — minimise = maximise entropy = prevent collapse

def compute_loss(train_centroids, heldout_centroids, df,
                 lambda1=1.0, lambda2=1.0, lambda3=0.1):

    # L_proximity
    prox_terms = []
    for concept, centroid in train_centroids.items():
        sub = df[(df['concept'] == concept) & (df['split'] == 'train')]
        for _, row in sub.iterrows():
            w = SPEAKER_WEIGHTS[row['lang']] / sum(SPEAKER_WEIGHTS.values())
            prox_terms.append(w * cosine_dist(centroid, row['emb']))
    L_proximity = float(np.sum(prox_terms))

    # L_coverage
    vocab_centroids = np.stack(list(train_centroids.values()))
    cov_dists = []
    for concept, hc in heldout_centroids.items():
        dists = [cosine_dist(hc, vc) for vc in vocab_centroids]
        cov_dists.append(min(dists))
    L_coverage = float(np.mean(cov_dists))

    # L_complexity (inverse entropy of pairwise distance distribution)
    pw_dists = pdist(vocab_centroids, metric='cosine')
    # bin into histogram to compute entropy
    counts, _ = np.histogram(pw_dists, bins=20)
    counts     = counts[counts > 0].astype(float)
    probs      = counts / counts.sum()
    H          = -float(np.sum(probs * np.log(probs + 1e-12)))
    L_complexity = 1.0 / (H + 1e-9)

    total = lambda1 * L_proximity + lambda2 * L_coverage + lambda3 * L_complexity

    return {
        'L_proximity'   : round(L_proximity,  6),
        'L_coverage'    : round(L_coverage,   6),
        'L_complexity'  : round(L_complexity, 6),
        'H_pairwise'    : round(H,             4),
        'L_total'       : round(total,         6),
    }

losses = compute_loss(train_centroids, heldout_centroids, df)
print('Universal Language Loss:')
for k, v in losses.items():
    print(f'  {k:<20} {v}')

# Pareto scan: vary λ1 vs λ2 to see the tension
print('\nPareto scan (λ1 vs λ2, λ3=0.1):')
print(f'{"λ1":>6} {"λ2":>6} {"L_prox":>10} {"L_cov":>10} {"L_total":>10}')
for l1 in [0.1, 0.5, 1.0, 2.0, 5.0]:
    for l2 in [0.1, 1.0, 5.0]:
        r = compute_loss(train_centroids, heldout_centroids, df, l1, l2, 0.1)
        print(f'{l1:>6.1f} {l2:>6.1f} {r["L_proximity"]:>10.5f} {r["L_coverage"]:>10.5f} {r["L_total"]:>10.5f}')

In [ ]:
# ── Compositionality probe ────────────────────────────────────────────────────
# Test: centroid(A) - centroid(B) + centroid(C) ≈ centroid(D)
# Measure: rank of the true D among all concepts by cosine similarity

def analogy_probe(a, b, c, expected_d, centroids, top_k=5):
    """Semantic arithmetic: a - b + c → expected_d"""
    if any(x not in centroids for x in [a, b, c, expected_d]):
        return None
    query = centroids[a] - centroids[b] + centroids[c]
    query /= np.linalg.norm(query + 1e-12)

    sims = {}
    for name, vec in centroids.items():
        if name not in [a, b, c]:
            sims[name] = float(np.dot(query, vec))

    ranked = sorted(sims.items(), key=lambda x: -x[1])
    rank   = next((i+1 for i, (n, _) in enumerate(ranked) if n == expected_d), None)
    top    = ranked[:top_k]
    return {'rank': rank, 'top': top, 'expected': expected_d}

# Use all 40 concepts (train + heldout) for richer probe
all_centroids = {**train_centroids, **heldout_centroids}

ANALOGY_TESTS = [
    # (A, B, C, expected_D)  — A - B + C ≈ D
    ('light',  'dark',   'love',   'fear'),   # opposites transfer
    ('dark',   'light',  'fear',   'love'),
    ('give',   'take',   'speak',  'think'),  # action pairs
    ('life',   'death',  'sleep',  'dream'),  # abstract
    ('war',    'peace',  'pain',   'joy'),    # held-out antonyms
    ('friend', 'enemy',  'hope',   'fear'),
    ('find',   'lose',   'build',  'break'),
    ('water',  'fire',   'river',  'forest'), # physical
]

print('Compositionality probe: A − B + C ≈ D')
print(f'{"A":<10} {"B":<10} {"C":<10} {"Expected D":<12} {"Rank":<6} {"Top-3 results"}')
print('-' * 85)

ranks = []
for a, b, c, d in ANALOGY_TESTS:
    r = analogy_probe(a, b, c, d, all_centroids)
    if r is None:
        print(f'{a:<10} {b:<10} {c:<10} {d:<12} MISSING')
        continue
    top3 = ', '.join(f'{n}({s:.3f})' for n, s in r['top'][:3])
    print(f'{a:<10} {b:<10} {c:<10} {d:<12} {str(r["rank"]):<6} {top3}')
    ranks.append(r['rank'])

if ranks:
    mrr = float(np.mean([1.0/r for r in ranks]))
    hits1 = sum(1 for r in ranks if r == 1) / len(ranks)
    hits3 = sum(1 for r in ranks if r <= 3) / len(ranks)
    print(f'\nMRR={mrr:.3f}  Hits@1={hits1:.2f}  Hits@3={hits3:.2f}')
    print('MRR > 0.5 suggests compositional structure in the concept space.')

In [ ]:
# ── UMAP visualisation ────────────────────────────────────────────────────────
if HAS_UMAP:
    import umap as umap_lib

    # Project: all word embeddings + all centroids
    train_df = df[df['split'] == 'train']
    train_embs_arr = np.stack(train_df['emb'].values)

    all_c_matrix = np.stack([all_centroids[c] for c in TRAIN_CONCEPTS + HELD_OUT_CONCEPTS])
    to_project   = np.vstack([train_embs_arr, all_c_matrix])

    reducer   = umap_lib.UMAP(n_components=2, random_state=42,
                               metric='cosine', n_neighbors=8, min_dist=0.1)
    projected = reducer.fit_transform(to_project)

    n_words       = len(train_embs_arr)
    word_proj     = projected[:n_words]
    centroid_proj = projected[n_words:]

    fig, axes = plt.subplots(1, 2, figsize=(20, 8))

    # Left: languages coloured
    COLORS = {'en':'#E24B4A','zh':'#FF9800','hi':'#9C27B0','es':'#378ADD',
              'ar':'#009688','ru':'#795548','pt':'#D4537E','fr':'#EF9F27',
              'de':'#1D9E75','ja':'#7F77DD'}
    for lang in MULTILINGUAL_VOCAB:
        mask = train_df['lang'].values == lang
        axes[0].scatter(word_proj[mask, 0], word_proj[mask, 1],
                        c=COLORS[lang], s=35, alpha=0.6, label=lang, zorder=2)
    # Train centroids
    axes[0].scatter(centroid_proj[:len(TRAIN_CONCEPTS), 0],
                    centroid_proj[:len(TRAIN_CONCEPTS), 1],
                    c='black', s=160, marker='*', zorder=4, label='universal centroid')
    for i, c in enumerate(TRAIN_CONCEPTS):
        axes[0].annotate(c, (centroid_proj[i,0]+0.03, centroid_proj[i,1]+0.03), fontsize=7, fontweight='bold')
    axes[0].set_title('10-language concept space\n(stars = proto-tokens for universal language)', fontsize=11)
    axes[0].legend(title='Language', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
    axes[0].axis('off')

    # Right: quality score as circle size
    qs = quality_df.set_index('concept')['quality_score']
    sizes = np.array([qs.get(c, 1.0) * 50 for c in TRAIN_CONCEPTS])
    sc = axes[1].scatter(centroid_proj[:len(TRAIN_CONCEPTS), 0],
                         centroid_proj[:len(TRAIN_CONCEPTS), 1],
                         c=quality_df.sort_values('quality_score')['quality_score'].reindex(
                             quality_df.set_index('concept').loc[TRAIN_CONCEPTS].index).values,
                         s=sizes, cmap='RdYlGn', zorder=3, alpha=0.85)
    # Held-out centroids
    axes[1].scatter(centroid_proj[len(TRAIN_CONCEPTS):, 0],
                    centroid_proj[len(TRAIN_CONCEPTS):, 1],
                    c='gray', s=60, marker='D', alpha=0.5, zorder=2, label='held-out')
    for i, c in enumerate(TRAIN_CONCEPTS):
        axes[1].annotate(c, (centroid_proj[i,0]+0.03, centroid_proj[i,1]+0.03), fontsize=7)
    for i, c in enumerate(HELD_OUT_CONCEPTS):
        axes[1].annotate(c, (centroid_proj[len(TRAIN_CONCEPTS)+i,0]+0.03,
                             centroid_proj[len(TRAIN_CONCEPTS)+i,1]+0.03), fontsize=6, color='gray')
    plt.colorbar(sc, ax=axes[1], label='quality score')
    axes[1].set_title('Proto-token quality (size + colour = quality score)\nGrey diamonds = held-out concepts', fontsize=11)
    axes[1].axis('off')

    plt.suptitle('Universal Language Bootstrapper v2 — 10 Languages, 40 Concepts', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('exp7_v2_concept_space.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Plot saved.')
else:
    print('UMAP not available. Install with: pip install umap-learn')

In [ ]:
# ── Quality gate summary ──────────────────────────────────────────────────────
QUALITY_THRESHOLD = 2.0
COVERAGE_THRESHOLD = 0.15   # intra_variance must be below this

passed = quality_df[
    (quality_df['quality_score']  >= QUALITY_THRESHOLD) &
    (quality_df['intra_variance'] <= COVERAGE_THRESHOLD)
]
failed = quality_df[
    (quality_df['quality_score']  <  QUALITY_THRESHOLD) |
    (quality_df['intra_variance'] >  COVERAGE_THRESHOLD)
]

print(f'Quality gate (score≥{QUALITY_THRESHOLD}, intra_var≤{COVERAGE_THRESHOLD}):')
print(f'  PASSED ({len(passed)}): {list(passed["concept"])}' )
print(f'  FAILED ({len(failed)}): {list(failed["concept"])}' )
print()
print('Interpretation:')
print('  Passed = safe to assign a universal token (physically concrete concepts tend to pass)')
print('  Failed = abstract concepts with high cross-linguistic variance (need multi-token expressions)')

# Save results
quality_df.to_csv('exp7_v2_quality_scores.csv', index=False)
pd.DataFrame([losses]).to_csv('exp7_v2_loss.csv', index=False)
print('\nResults saved to exp7_v2_quality_scores.csv and exp7_v2_loss.csv')